<a href="https://colab.research.google.com/github/Vanquan99/Tool_Copy-Google-Drive/blob/main/tool_coppy_driver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Drive All-in-One Tool (Copy / Check Size / Unzip)
Chọn `action`, điền `source_link` (có thể nhập nhiều link cách nhau bằng dấu phẩy `,`), `dest_link` và bấm **Run** 1 lần duy nhất.

In [ ]:
#@title Google Drive All-in-One (Điền thông tin & bấm Run 1 lần)
action = "Copy" #@param ["Copy", "Check Size (Tính dung lượng)", "Unzip (Giải nén)"]
# Có thể nhập nhiều link hoặc ID nguồn, cách nhau bằng dấu phẩy (,)
source_link = "" #@param {type:"string"}
# Link hoặc ID thư mục đích (không bắt buộc khi chọn Check Size)
dest_link = "" #@param {type:"string"}
workers = 4 #@param {type:"slider", min:1, max:8, step:1}
from_page = 0 #@param {type:"integer"}
to_page = 0 #@param {type:"integer"}
max_download_size_gb = 700 #@param {type:"number"}
exclude_str = "" #@param {type:"string"}
use_progress = True #@param {type:"boolean"}

import io, os, re, shutil, tempfile, threading, time, zipfile, tarfile
from concurrent.futures import ThreadPoolExecutor, as_completed
import httplib2
import google_auth_httplib2
from google.auth import default
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from tqdm.notebook import tqdm

FOLDER_MIME = 'application/vnd.google-apps.folder'

def extract_id(val):
    m = re.search(r'[-\w]{25,}', val or '')
    return m.group(0) if m else (val or '').strip()

def esc(s):
    return s.replace('\\', '\\\\').replace("'", "\\'")

src_ids = [extract_id(x) for x in source_link.split(',') if extract_id(x)]
dst_id = extract_id(dest_link)
if not src_ids:
    raise ValueError('Vui lòng nhập ít nhất 1 source_link.')
if not action.startswith('Check Size') and not dst_id:
    raise ValueError('Vui lòng nhập dest_link cho tác vụ Copy / Unzip.')

auth.authenticate_user()
creds, _ = default()
_tls = threading.local()

def get_service():
    # shortcut: mỗi thread giữ 1 kết nối httplib2 riêng vì httplib2 không thread-safe
    if not hasattr(_tls, 'service'):
        http = google_auth_httplib2.AuthorizedHttp(creds, http=httplib2.Http(timeout=120))
        _tls.service = build('drive', 'v3', http=http, cache_discovery=False)
    return _tls.service

def api_call(fn, retries=4):
    for i in range(retries):
        try:
            return fn()
        except Exception:
            if i == retries - 1:
                raise
            time.sleep((2 ** i) + 0.2)

excluded = [x.strip() for x in exclude_str.split(',') if x.strip()]
limit_mb = float(max_download_size_gb) * 1024 if max_download_size_gb > 0 else float('inf')
total_mb = 0.0
stop_flag = False
_lock = threading.Lock()

def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    with _lock:
        print(line)
        with open('copy_log.txt', 'a', encoding='utf-8') as f:
            f.write(line + '\n')

def list_dest_map(folder_id):
    items, token = {}, None
    while True:
        res = api_call(lambda: get_service().files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields='files(id, name, mimeType), nextPageToken',
            pageSize=1000, pageToken=token,
            supportsAllDrives=True, includeItemsFromAllDrives=True
        ).execute())
        for f in res.get('files', []):
            items[f['name']] = f
        token = res.get('nextPageToken')
        if not token:
            break
    return items

def get_or_create_folder(parent_id, name, dest_map=None):
    existing = (dest_map or {}).get(name)
    if existing and existing.get('mimeType') == FOLDER_MIME:
        return existing['id']
    if dest_map is None:
        res = api_call(lambda: get_service().files().list(
            q=f"'{parent_id}' in parents and name = '{esc(name)}' and mimeType = '{FOLDER_MIME}' and trashed = false",
            fields='files(id)', supportsAllDrives=True, includeItemsFromAllDrives=True
        ).execute()).get('files', [])
        if res:
            return res[0]['id']
    created = api_call(lambda: get_service().files().create(
        body={'name': name, 'mimeType': FOLDER_MIME, 'parents': [parent_id]},
        fields='id', supportsAllDrives=True
    ).execute())
    if dest_map is not None:
        dest_map[name] = {'id': created['id'], 'name': name, 'mimeType': FOLDER_MIME}
    return created['id']

def get_children(folder_id, fp=0, tp=0):
    files, token, page = [], None, 0
    q = f"'{folder_id}' in parents and trashed = false"
    if excluded:
        q += ' and (' + ' and '.join(f"not name contains '{esc(x)}'" for x in excluded) + ')'
    while True:
        res = api_call(lambda: get_service().files().list(
            q=q, orderBy='name, createdTime',
            fields='files(id, name, mimeType, size), nextPageToken',
            pageSize=1000, pageToken=token,
            supportsAllDrives=True, includeItemsFromAllDrives=True
        ).execute())
        page += 1
        if (fp < page <= tp) or tp == 0:
            files.extend(res.get('files', []))
        token = res.get('nextPageToken')
        if not token or (tp > 0 and page >= tp):
            break
    return files

def copy_one_file(parent_id, meta, dest_map):
    global total_mb, stop_flag
    if stop_flag:
        return
    if meta['name'] in dest_map:
        log(f"{meta['name']} - Đã tồn tại, bỏ qua")
        return
    size_mb = int(meta.get('size', 0)) / (1024 * 1024) if meta.get('size') else 0.0
    with _lock:
        if total_mb + size_mb > limit_mb:
            if not stop_flag:
                stop_flag = True
                log(f"Đạt giới hạn dung lượng ({max_download_size_gb} GB), dừng sao chép.")
            return
        total_mb += size_mb
        curr_gb = total_mb / 1024
    t0 = time.time()
    api_call(lambda: get_service().files().copy(
        fileId=meta['id'], body={'parents': [parent_id]}, supportsAllDrives=True
    ).execute())
    log(f"[{meta['name']}] Đã sao chép {size_mb:.2f} MB ({time.time() - t0:.2f}s) | Tổng: {curr_gb:.2f} GB")

def copy_tree(dest_folder_id, items):
    if stop_flag:
        return
    dest_map = list_dest_map(dest_folder_id)
    folders = [x for x in items if x.get('mimeType') == FOLDER_MIME]
    files = [x for x in items if x.get('mimeType') != FOLDER_MIME]
    if files:
        with ThreadPoolExecutor(max_workers=max(1, int(workers))) as ex:
            futs = [ex.submit(copy_one_file, dest_folder_id, f, dest_map) for f in files]
            it = tqdm(as_completed(futs), total=len(futs), desc='Đang sao chép') if use_progress else as_completed(futs)
            for fut in it:
                fut.result()
    for folder in folders:
        if stop_flag:
            break
        log(f"Vào thư mục: {folder['name']}")
        sub_id = get_or_create_folder(dest_folder_id, folder['name'], dest_map)
        copy_tree(sub_id, get_children(folder['id']))

def check_size(item_id):
    meta = api_call(lambda: get_service().files().get(
        fileId=item_id, supportsAllDrives=True, fields='id,name,mimeType,size'
    ).execute())
    if meta.get('mimeType') != FOLDER_MIME:
        sz = int(meta.get('size', 0))
        log(f"File: {meta['name']} | {sz / (1024**2):.2f} MB ({sz / (1024**3):.2f} GB)")
        return 1, 0, sz
    files_cnt, folders_cnt, total_bytes = 0, 1, 0
    stack = [item_id]
    while stack:
        fid = stack.pop()
        for ch in get_children(fid):
            if ch.get('mimeType') == FOLDER_MIME:
                folders_cnt += 1
                stack.append(ch['id'])
            else:
                files_cnt += 1
                total_bytes += int(ch.get('size', 0))
    log(f"Thư mục: {meta['name']} | {files_cnt} files, {folders_cnt} folders | {total_bytes / (1024**2):.2f} MB ({total_bytes / (1024**3):.2f} GB)")
    return files_cnt, folders_cnt, total_bytes

def upload_local_dir(local_dir, parent_drive_id):
    dest_map = list_dest_map(parent_drive_id)
    entries = sorted(os.listdir(local_dir))
    files = [e for e in entries if os.path.isfile(os.path.join(local_dir, e))]
    dirs = [e for e in entries if os.path.isdir(os.path.join(local_dir, e))]

    def _up(fname):
        if fname in dest_map:
            log(f"{fname} - Đã tồn tại, bỏ qua")
            return
        fpath = os.path.join(local_dir, fname)
        sz_mb = os.path.getsize(fpath) / (1024 * 1024)
        media = MediaFileUpload(fpath, resumable=True)
        api_call(lambda: get_service().files().create(
            body={'name': fname, 'parents': [parent_drive_id]},
            media_body=media, fields='id', supportsAllDrives=True
        ).execute())
        log(f"[Upload] {fname} ({sz_mb:.2f} MB)")

    if files:
        with ThreadPoolExecutor(max_workers=max(1, int(workers))) as ex:
            for fut in as_completed([ex.submit(_up, f) for f in files]):
                fut.result()
    for d in dirs:
        sub_id = get_or_create_folder(parent_drive_id, d, dest_map)
        upload_local_dir(os.path.join(local_dir, d), sub_id)

def unzip_drive_file(file_id, dest_folder_id):
    meta = api_call(lambda: get_service().files().get(
        fileId=file_id, supportsAllDrives=True, fields='id,name,mimeType,size'
    ).execute())
    name = meta['name']
    tmp_dir = tempfile.mkdtemp(prefix='gdrive_unzip_')
    archive_path = os.path.join(tmp_dir, name)
    extract_dir = os.path.join(tmp_dir, 'out')
    os.makedirs(extract_dir, exist_ok=True)
    try:
        log(f"Đang tải file nén: {name}...")
        req = get_service().files().get_media(fileId=file_id, supportsAllDrives=True)
        with io.FileIO(archive_path, 'wb') as fh:
            downloader = MediaIoBaseDownload(fh, req, chunksize=32 * 1024 * 1024)
            done = False
            while not done:
                _, done = downloader.next_chunk()
        log(f"Đang giải nén: {name}...")
        if zipfile.is_zipfile(archive_path):
            with zipfile.ZipFile(archive_path, 'r') as zf:
                zf.extractall(extract_dir)
        elif tarfile.is_tarfile(archive_path):
            with tarfile.open(archive_path, 'r:*') as tf:
                tf.extractall(extract_dir)
        else:
            shutil.unpack_archive(archive_path, extract_dir)
        base_folder = re.sub(r'\.(zip|tar\.gz|tgz|tar\.bz2|tar|7z|rar)$', '', name, flags=re.I)
        target_id = get_or_create_folder(dest_folder_id, base_folder)
        log(f"Đang tải nội dung giải nén lên thư mục: {base_folder}...")
        upload_local_dir(extract_dir, target_id)
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

log(f"Bắt đầu tác vụ: {action} ({len(src_ids)} nguồn)")
if action.startswith('Check Size'):
    tot_f, tot_d, tot_b = 0, 0, 0
    for sid in src_ids:
        f, d, b = check_size(sid)
        tot_f += f; tot_d += d; tot_b += b
    if len(src_ids) > 1:
        log(f"==> TỔNG CỘNG: {tot_f} files, {tot_d} folders | {tot_b / (1024**3):.2f} GB")
elif action.startswith('Unzip'):
    for sid in src_ids:
        unzip_drive_file(sid, dst_id)
    log('Hoàn tất giải nén!')
else:
    for sid in src_ids:
        if stop_flag:
            break
        meta = api_call(lambda: get_service().files().get(
            fileId=sid, supportsAllDrives=True, fields='id,name,mimeType,size'
        ).execute())
        if meta.get('mimeType') != FOLDER_MIME:
            copy_one_file(dst_id, meta, list_dest_map(dst_id))
        else:
            target_id = get_or_create_folder(dst_id, meta['name'])
            copy_tree(target_id, get_children(sid, from_page, to_page))
    log(f"Hoàn tất sao chép. Tổng dung lượng đã copy: {total_mb / 1024:.2f} GB")